<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap6_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

６章言語モデルのファインチューニング

- ファインチューニング済みエンコーダーモデルを用いたテキストのトピック分類
- 現代の LLM 時代におけるエンコーダーベースモデルの役割の理解
- デコーダーモデルを使った特定のスタイルのテキスト生成
- 命令型ファインチューニングによる単一モデルでの複数タスクの解決
- 小さいGPUでもモデルを訓練できるパラメーター効率の高いファインチューニング手法
- よりすくに計算資源でモデルの推論を実行できる手法

6.1.1 データセットの特定

In [2]:
%pip install genaibook

In [3]:
from datasets import load_dataset

#az news データセットはテキスト分類モデルのベンチマークやデータマイニング、情報検索、データストリーミングなどの研究で広く用いられている。
# 訓練用のサンプルは１２万件あり、ファインにチューニングには十分
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [4]:
# データの具体的な例を見ていこう

raw_train_datasets = raw_datasets["train"]
raw_train_datasets[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

//{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

サンプルにテキストとラベルが含まれているが、２はどのクラスを指しているのか？
これを知るにはデータセットの features とその label フィールドを見ればいい。

In [5]:
print(raw_train_datasets.features)

# results
# {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
# 0なら世界のニュース、1ならスポーツ、2ならビジネス、３なら科学技術のニュース

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


# 6.1.2 使用するモデルタイプの定義

## Transformer おさらい

- エンコーダーモデル：入力の意味表現を捉える
- デコーダーモデル：文章などの新しいシーケンスを出力することを目的にしたモデル。テキスト生成に最適。
- エンコーダーデコーダー型モデル：入力シーケンスを異なる出力シーケンスに変換するタスクに適している

今回は、**分類ヘッド付きエンコーダーモデル**をアプローチとして採用する。
エンコーダーモデルにシンプルな分類ネットワーク（ヘッド）を埋め込みに追加してファインチューニングする方法。

ベースモデルの要件は以下の４つ

- エンコーダベースであること
- GPU を使えば数分くらいでファインチューニングできるモデル
- 事前訓練で確かな成果を残しているもの
- 短いテキストシーケンスを処理できること

DistilBERT が良さげらしい。

6.1.4 データセットの前処理

トークナイザーは AutoTokenizer を使おう。
transformers ライブラリは入力の長さがすべて同じ出なければならないので、padding=true で使おう。


In [6]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(batch):
  return tokenizer(
      batch["text"], truncation=True, padding=True, return_tensors="pt"
  )

tokenize_function(raw_train_datasets[:2])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': tensor([[  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
          1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813,
          2395,  1005,  1055,  1040, 11101,  2989,  1032,  2316,  1997, 11087,
          1011, 22330,  8713,  2015,  1010,  2024,  3773,  2665,  2153,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101, 18431,  2571,  3504,  2646,  3293, 13395,  1006, 26665,  1007,
         26665,  1011,  2797,  5211,  3813, 18431,  2571,  2177,  1010,  1032,
          2029,  2038,  1037,  5891,  2005,  2437,  2092,  1011, 22313,  1998,
          5681,  1032,  6801,  3248,  1999,  1996,  3639,  3068,  1010,  2038,
          5168,  2872,  1032,  2049, 29475,  2006,  2178,  2112,  1997,  1996,
          3006,  1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
# データセット内の各要素に対して関数を並列で適用するもの

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

6.1.5 評価指標の定義

評価指標には evaluate というライブラリが使える。
文章分類の場合は以下の指標が有力な候補となる。

- 正解率
- 適合率
- 再現率
- F1 スコア

evaluate が提供する指標には compute() メソッドがある。

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
print(accuracy.description)
print(accuracy.compute(references=[0, 1, 0, 1], predictions=[1, 0, 0, 1]))

In [ ]:
f1_score = evaluate.load("f1")

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  # 精度と F1 スコアを求める
  acc_result = accuracy.compute(references=labels, predictions=preds)
  acc = acc_result["accuracy"]

  f1_result = f1_score.compute(
      references=labels, predictions=preds, average="weighted"
  )
  f1 = f1_result["f1"]

  return {"accuracy": acc, "f1": f1}

6.1.6 モデルの訓練

DistilBERT はエンコーダーモデルなので、そのまま使うと埋め込みが得られるだけなので、分類タスクには使えない。
この埋め込みを分類ヘッドに渡す必要がある。

AutoModelForSequenceClassification でモデルを読み込み、分類ヘッドでモデルを訓練する。以下の２つの処理が行われる。
- 言語モデルのヘッドを取り外して読み込む。ここはモデルのエンコーダー部分であり、各トークンに対して埋め込みを出力する
- モデルの上にランダムに初期化された分類ヘッドを追加する。このヘッドは単なる線形層であり、プーリング埋め込みを受け取り、クラスごとの確率を出力する。

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

from genaibook.core import get_device

device = get_device()
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels
).to(device)

モデルの初期化ができたのでいよいよ訓練開始。

In [ ]:
from transformers import TrainingArguments
from google.colab import userdata # userdataをインポート

batch_size = 32
training_args = TrainingArguments(
    "classifier-chapter4",
    push_to_hub=True, #モデルが保存されるたびに HuggingFace にプッシュするかどうか
    num_train_epochs=2, #何回転させるか
    eval_strategy="epoch", # 評価するタイミングの指定（epoch 終了時を指定）
    per_device_train_batch_size=batch_size, # 訓練時のコアあたりのバッチサイズ
    per_device_eval_batch_size=batch_size,
    hub_token=userdata.get('HF_TOKEN') # シークレットからトークンを取得して使用
)

In [ ]:
#AG News データセットからデータを取得して訓練開始。

from transformers import Trainer
# huggingface_hub import notebook_login は削除。

# notebook_login() は不要になるため削除。

# データセットをシャッフルし訓練用に１万件のサンプルを抽出する
shuffled_dataset = tokenized_datasets["train"].shuffle(seed=42)
small_split = shuffled_dataset.select(range(10000))

# Trainer を初期化する
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=small_split,
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

In [ ]:
import os
from google.colab import userdata

# Set the WANDB API key from Colab secrets
os.environ["WANDB_API_KEY"] = userdata.get('WADB')

In [ ]:
# trainer を初期化し訓練開始

trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
# パイプラインを使って高レベルに操作する
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model="shown5/classifier-chapter4",
    device=device
)
pipe(
    """The soccer match between Sapin and Portugal ended in a terrible result for Portugal."""
)

In [ ]:
# 予測結果の評価

# すべてのサンプルについて予測を得る
model_preds = pipe(list(tokenized_datasets["test"]["text"])) # ここを修正

# データセットのラベルを得る
references = tokenized_datasets["test"]["label"]


# ラベルのリストを得る
label_names = raw_train_datasets.features["label"].names

# 最初の３サンプルの結果を表示する
samples = 3
texts = tokenized_datasets["test"]["text"][:samples]
for pred, ref, text in zip(model_preds[:samples], references[:samples], texts):
  print(f"Predicted: {pred['label']}; Actual {label_names[ref]};")
  print(text)

In [ ]:
# 混同行列（真陽性偽陽性など）でモデルの性能を確認する

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# 予測されたラベルをIDに変換する
label_to_id = {name: i for i, name in enumerate(label_names)}
preds_labels = [int(pred["label"].split('_')[1]) for pred in model_preds] # ここを修正

#混同行列を求める
confusion_matrix = evaluate.load("confusion_matrix")
cm = confusion_matrix.compute(
    references=references,
    predictions=preds_labels,
    normalize="true"
)["confusion_matrix"]

# 混同行列をプロットする
fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
plt.title("Normalised confusion matrix")
plt.show()

6.1.7 今でも役に立つのか

以前として小型のカスタム分類機が役に立つことはある。とりわけ速度と効率が重要なアプリケーションでは。
例えば超大規模言語モデルの訓練データの準備で有用。大量の訓練データを大規模モデルに投入するのはコストが高い。
また、検索システム向けに埋め込みを得ることにも使える。
とはいえ、やはり高性能モデルの利用に移行しつつあるのが現状。

6.2 テキスト生成

In [ ]:
filtered_datasets = raw_datasets.filter(lambda example: example["label"] == 2)
filtered_datasets = filtered_datasets.remove_columns("label")

6.2.1 適切な生成モデルの選択

どのベースモデルを使うのかを判断するための要素は次のとおり。

- モデルサイズ：デカすぎても小さすぎてもダメ。
- 訓練データ：推論時のデータと類似している訓練データを選定すべし
- コンテキスト長：長文を生成する際にはコンテキストも長いものにする必要がある
- ライセンス：商用OKかどうかは確認すべし


6.2.2 生成モデルの訓練

In [ ]:
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

model_id = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = (
    tokenizer.eos_token
)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

In [ ]:
# データセットをトークン化する

def tokenize_function(batch):
  return tokenizer(batch["text"], truncation=True)

tokenized_datasets = filtered_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"], # input_ids と attention_mask のみ必要
)

In [ ]:
tokenized_datasets

In [ ]:
# 因果言語モデリングようのデータコレーターを作成する

from transformers import DataCollatorForLanguageModeling

# mlm はマスク言語モデルの略
# 今回はマスク言語モデルを訓練するわけではなく、因果言語モデル（causal language model）を訓練するためにFalseに設定する
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
samples = [tokenized_datasets["train"][i] for i in range(3)]
for sample in samples:
  print(f"input_ids shape: {len(sample['input_ids'])}")

In [ ]:
out = data_collator(samples)
for key in out:
  print(f"{key} shape : {out[key].shape}")

In [ ]:
training_args = TrainingArguments(
    "business-news-generator",
    push_to_hub=True,
    per_device_train_batch_size=8,
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=200
)

In [ ]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets["train"].select(range(5000)),
    eval_dataset=tokenized_datasets["test"],
)

In [ ]:
trainer.train()

In [ ]:
trainer.push_to_hub()

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="shown5/business-news-generator",
    device=device
)
print(
    pipe("Q1", do_sample=True, temperature=0.1, max_new_tokens=30)[0]["generated_text"]
)
print(
    pipe("Wall", do_sample=True, temperature=0.1, max_new_tokens=30)[0]["generated_text"]
)
print(
    pipe("Google", do_sample=True, temperature=0.1, max_new_tokens=30)[0]["generated_text"]
)

6.3 インストラクション